# CSIRO Biomass - Inference for P100 (16GB VRAM)

**最適化**: P100用にメモリ使用量を削減

In [ ]:
import os
import gc
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# P100最適化設定
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = False  # P100はTF32非対応

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu} ({vram:.1f} GB)")
    print(f"Free memory: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

In [ ]:
class CFG:
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    DATA_DIR = Path("/kaggle/input/csiro-biomass")
    MODEL_DIR = Path("/kaggle/input/csiro-dinov3-swa-v1")  # v1モデル
    
    # P100用に縮小
    IMG_SIZE = 384  # 512から縮小
    N_FOLDS = 5
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    BATCH_SIZE = 1
    NUM_WORKERS = 0
    
    # TTA設定（軽量化）
    USE_TTA = False  # P100ではTTAを無効化
    USE_FP16 = True  # FP16で推論

In [ ]:
# Load test data
test_df = pd.read_csv(CFG.DATA_DIR / "test.csv")
print(f"Test samples: {len(test_df)}")
test_wide = test_df[["image_path"]].drop_duplicates().reset_index(drop=True)
print(f"Unique test images: {len(test_wide)}")

In [ ]:
# Model definition (同じ構造)
class LocalMambaBlock(nn.Module):
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size//2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        x = x * torch.sigmoid(self.gate(x))
        x = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        x = self.proj(x)
        return shortcut + self.drop(x)


class BiomassModel(nn.Module):
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="")
        nf = self.backbone.num_features
        
        # Gradient checkpointingは推論時は不要
        
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )

    def forward(self, x):
        left, right = x
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x = self.fusion(torch.cat([x_l, x_r], dim=1))
        x = self.pool(x.transpose(1, 2)).flatten(1)
        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        gdm = green + clover
        total = green + clover + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("✅ Model defined")

In [ ]:
# Dataset
class TestDataset(Dataset):
    def __init__(self, df, data_dir, transform):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["image_path"]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        left = img.crop((0, 0, w // 2, h))
        right = img.crop((w // 2, 0, w, h))
        left = self.transform(left)
        right = self.transform(right)
        return left, right, row["image_path"]

def collate_fn(batch):
    lefts = torch.stack([b[0] for b in batch])
    rights = torch.stack([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return lefts, rights, paths

# P100用の軽量変換
test_tfms = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Inference (メモリ最適化版)
test_dataset = TestDataset(test_wide, CFG.DATA_DIR, test_tfms)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=False  # P100では無効化
)

print(f"DataLoader: {len(test_dataset)} images")

FOLD_WEIGHTS = [1.0, 0.9, 1.0, 1.1, 0.9]  # 調整された重み
all_preds = []
all_paths = None

for fold in range(CFG.N_FOLDS):
    # EMAモデルのみ使用（SWAは省略）
    model_path = CFG.MODEL_DIR / f"best_ema_fold{fold}.pth"
    if not model_path.exists():
        print(f"Fold {fold}: skip - not found")
        continue
    
    print(f"\n{'='*40}")
    print(f"Fold {fold}: loading...")
    
    # モデルロード前にメモリクリア
    torch.cuda.empty_cache()
    gc.collect()
    
    model = BiomassModel(CFG.BACKBONE, pretrained=False)
    
    # CPU上でロードしてからGPUへ
    state_dict = torch.load(model_path, map_location="cpu")
    if list(state_dict.keys())[0].startswith("module."):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    # メモリ使用量確認
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        print(f"Memory allocated: {allocated:.1f} GB")
    
    preds = []
    paths = []
    
    with torch.no_grad():
        for i, (left, right, p) in enumerate(tqdm(test_loader, desc=f"Fold {fold}")):
            left = left.to(device)
            right = right.to(device)
            
            # FP16推論
            if CFG.USE_FP16:
                with torch.cuda.amp.autocast():
                    out = model((left, right))
            else:
                out = model((left, right))
            
            preds.append(out.cpu().numpy())
            paths.extend(p)
            
            # 定期的にメモリクリア
            if i % 50 == 0:
                torch.cuda.empty_cache()
    
    preds = np.vstack(preds)
    all_preds.append(preds * FOLD_WEIGHTS[fold])
    if all_paths is None:
        all_paths = paths
    
    print(f"Fold {fold}: done, shape={preds.shape}")
    
    # モデル削除とメモリクリア
    del model, state_dict
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*40}")
print(f"Total folds used: {len(all_preds)}")

In [ ]:
# Ensemble & Submission
total_weight = sum(FOLD_WEIGHTS[:len(all_preds)])
ensemble = np.sum(all_preds, axis=0) / total_weight
print(f"Ensemble: {ensemble.shape}")

# Create predictions
preds_wide = pd.DataFrame(ensemble, columns=CFG.TARGETS)
preds_wide.insert(0, 'image_path', all_paths)

# Convert to long format
preds_long = preds_wide.melt(
    id_vars=['image_path'],
    value_vars=CFG.TARGETS,
    var_name='target_name',
    value_name='target'
)

# Merge with test_df
submission = pd.merge(
    test_df[['sample_id', 'image_path', 'target_name']],
    preds_long,
    on=['image_path', 'target_name'],
    how='left'
)

submission = submission[['sample_id', 'target']]
submission['target'] = submission['target'].fillna(0.0).clip(lower=0)
submission = submission.sort_values('sample_id').reset_index(drop=True)

# Save
submission.to_csv("submission.csv", index=False)

print(f"\n✅ Saved: submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nStats:")
print(submission['target'].describe())